# U1.L1 — Coupling flows on 2D toys

**Session U1.L1** (lab, joint). Companion to the U1.L1 lecture notes, which explain
every step here at length; this notebook is the part you execute.

By the end you own three objects:

1. a **bijection** you wrote, `AffineCoupling`, with an inverse and a log-determinant
   in both directions;
2. **exact density evaluation**, `Flow.log_prob`, which is U1.T1's change-of-variables
   theorem as code; and
3. an **independent Jacobian witness**: an automatic-differentiation Jacobian,
   assembled per example in float64, against which your analytic log-determinant is
   checked.

Then you train the result on two toy datasets, look at what it learned, and check it
against a library. The last picture raises a question this notebook does not answer.

### How to read this notebook

Cells marked **TODO** are yours. Each TODO is followed by a **Checkpoint** cell that
either passes or tells you what is wrong. Everything else is given code: run it,
read it, and move on. If you are working through this on your own, the U1.L1 notes
carry the reasoning behind each block, and the solutions notebook,
`lab-u1l1-solutions.ipynb`, which the instructor releases, carries the filled-in cells.

**Everything runs on CPU.** The two training runs take about 70 s each on a laptop
core; the checkpoints are *relative* (improvement over the first epoch), so a slower
machine can set `EPOCHS = 250` below and still pass them.

## Block 0 — Orientation: what you are given, and the sanity ritual

### Given code

Three modules under `common/` arrive as **given code**. You do not need to have
attended the bootcamp sessions that built two of them; their contracts are all that
this lab uses, and the contracts are written out here.

| Module | Built in | What this lab uses |
|---|---|---|
| `common/train.py` | U0.L1, extended in U0.L2 | `set_seed(seed)` and `Trainer(model, opt, loss_fn, device, ckpt_dir, log_every, *, sched, tracker, config)` with `.fit(train_loader, val_loader, epochs)` |
| `common/toy2d.py` | **today** (block 1) | the three frozen generators and the two picture helpers |
| `common/coupling.py` | **today** (the reference you rebuild) | imported only at the end, to cross-check your own module |

The `Trainer` runs the five-step loop `forward -> loss -> zero_grad -> backward -> step`
(then `sched.step()` **per step**), evaluates on the validation loader under
`no_grad` at every epoch, writes `last.pt` every epoch and `best.pt` when the
validation loss improves, and logs to a CSV **and** to MLflow when asked. Its interface
was frozen for classifiers, which is why it expects `(x, y)` batches and reports an
accuracy. A flow has no labels: `y` will be a placeholder and the accuracy will read
`nan`. That is the interface being preserved, not a bug.

### Where the curves go

The course logs to **MLflow, deployed locally**: a SQLite file next to this notebook
(`sqlite:///mlflow.db`) plus a `mlartifacts/` directory. Both are portable — you can zip
a run directory and hand it to someone — and both are gitignored. The reason the store
is local is that an offline teaching bundle must stand alone; it is not a recommendation
of any backend for production. (The older `mlruns/` file store you may meet in tutorials
is deprecated in MLflow 3.15, which is why the course uses SQLite.) To look at the
curves after training:

```
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

One deliberate departure from the bootcamp wiring, with its reason. The `Trainer` can
log to MLflow **per step** (`tracker="mlflow"`), and for an image model that is the
right cadence. Here a step is a 256-point batch through a 26k-parameter network —
under a millisecond — while one SQLite write costs a few milliseconds. Measured on this
notebook's runs: per-step tracking made a 14 s training take 48 s. So the loop runs
without the tracker (its CSV log and checkpoints are still written to `runs/`), and
the two NLL curves (negative log-likelihood — the loss, on train and on validation)
are logged to MLflow **per epoch**, after the loop, together with
the resolved configuration and the tags. The run directory stands alone either way,
which is the rule that matters.

In [ ]:
import sys
from pathlib import Path

# The repository root, so `import common.train` works from labs/.
ROOT = Path.cwd().parent if Path.cwd().name == "labs" else Path.cwd()
sys.path.insert(0, str(ROOT))

import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from common.train import Trainer, set_seed

torch.set_num_threads(4)
set_seed(0)
DEVICE = "cpu"          # deliberate: the lab is CPU-sufficient throughout
EPOCHS = 500            # 8000 steps at batch 256; halve it on a slow machine

print("torch", torch.__version__)

### The sanity ritual, on a scalar toy

The course-wide ritual, performed in every lab: after writing anything that claims to
be a derivative, compare it with **finite differences** in float64 along **one fixed
unit direction** $v$,

$$
v^{\top}\nabla_\theta L
\quad\text{against}\quad
\frac{L(\theta+\varepsilon v)-L(\theta-\varepsilon v)}{2\varepsilon},
$$

accepted under a mixed tolerance $|d_{\mathrm{AD}} - d_{\mathrm{FD}}| \le
\texttt{atol} + \texttt{rtol}\cdot\max(|d_{\mathrm{AD}}|, |d_{\mathrm{FD}}|)$. Two
forward evaluations, whatever the number of parameters. Run it once here, on a toy
loss, so that the shape is fresh.

**Today's check is a sibling of this one.** The object under suspicion is not a
gradient but a *log-determinant*: you will compute $\log|\det J_\psi|$ analytically
as a sum of scales, and check it against a Jacobian that automatic differentiation
assembles **independently**, one example at a time.

In [ ]:
def directional_check(loss_fn, theta, eps=1e-6, atol=1e-8, rtol=1e-6):
    """One fixed unit direction; two extra evaluations; mixed tolerance."""
    theta = theta.detach().double().requires_grad_(True)
    g = torch.autograd.grad(loss_fn(theta), theta)[0]
    v = torch.randn_like(theta)
    v = v / v.norm()
    with torch.no_grad():
        d_ad = float(g.flatten() @ v.flatten())
        hi = float(loss_fn(theta + eps * v))
        lo = float(loss_fn(theta - eps * v))
    d_fd = (hi - lo) / (2 * eps)
    tol = atol + rtol * max(abs(d_ad), abs(d_fd))
    return d_ad, d_fd, abs(d_ad - d_fd) <= tol


toy_x = torch.randn(32, 3, dtype=torch.float64)
theta0 = torch.randn(3, 2, dtype=torch.float64)
d_ad, d_fd, ok = directional_check(lambda th: torch.tanh(toy_x @ th).pow(2).mean(), theta0)
print(f"autograd {d_ad:.10e}   finite differences {d_fd:.10e}   accepted={ok}")
assert ok

## Block 1 — `toy2d`: the datasets, frozen today

`common/toy2d.py` is **infrastructure, not a TODO**: it is distributed complete, and it
is frozen from today. The reason is a contract with three later sessions. U2.L2 and
U3.L1 train other models on *exactly these points* and compare them with what you build
today; the comparison means something only if the data generator and the seeds are the
same. So downstream sessions import this module by name and never redefine a generator.

::: contract
**Artifact `toy2d-datasets` — frozen contract**

```
two_moons(n, seed)     -> Tensor (n, 2), float32
checkerboard(n, seed)  -> Tensor (n, 2), float32
rings(n, seed)         -> Tensor (n, 2), float32
EXTENT = (-4.0, 4.0)   # default plotting extent, both axes
density_heatmap(log_prob_fn, extent=EXTENT, n=200, ax=None) -> (xs, ys, logp)
warped_grid(generative_fn, extent=EXTENT, ax=None, ...)     -> lines
```

1. **Pure, seeded functions.** Same `seed`, same pinned environment, same tensor,
   bit for bit.
2. **Float32 outputs**, shape `(n, 2)`.
3. **The checkerboard has literal zero-density gaps.** It is an equal-weight mixture
   of **eight separated squares** of side $1.5$, centred in the black cells of a
   $4\times4$ board on $[-4,4]^2$, with **no Gaussian jitter**. The nearest corners of
   two same-colour squares are $0.5\sqrt{2} \approx 0.71$ apart, and the white cells
   carry no mass at all. The support of $q$ is a union of eight disjoint closed
   squares.
:::

Property 3 is not cosmetic. U1.T2 makes a statement about *literal* support, and this
generator is the witness it will use. Today you only look at what a fitted density
does in the gaps.

In [ ]:
from common.toy2d import (EXTENT, GENERATORS, checkerboard, density_heatmap,
                          in_checkerboard_support, rings, two_moons, warped_grid)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, (name, gen) in zip(axes, GENERATORS.items()):
    pts = gen(4096, 0).numpy()
    ax.scatter(pts[:, 0], pts[:, 1], s=3, color="#003366", alpha=0.5)
    ax.set_xlim(*EXTENT); ax.set_ylim(*EXTENT); ax.set_aspect("equal")
    ax.set_title(f"{name}(4096, seed=0)")
plt.tight_layout()
plt.show()

**Checkpoint 1** — the contract, executed: bit-identical across calls, seed-sensitive,
and the checkerboard never leaves its eight squares.

In [ ]:
for name, gen in GENERATORS.items():
    a, b = gen(4096, 0), gen(4096, 0)
    assert a.dtype == torch.float32 and a.shape == (4096, 2)
    assert torch.equal(a, b), f"{name}: not bit-identical"
    assert not torch.equal(a, gen(4096, 1)), f"{name}: seed ignored"
cb = checkerboard(100_000, 0)
assert bool(in_checkerboard_support(cb).all()), "a checkerboard point left the squares"
print("Checkpoint 1 passed: three seeded generators; 100,000 checkerboard points, "
      "0 outside the eight squares.")

## Block 2 — An affine coupling layer, from scratch

The layer of U1.T1 §5, written in the **generative direction** $\psi \colon z \to x$.
Split the coordinates into an untouched half $x_A$ (the conditioner reads it) and a
moved half $x_B$:

$$
\text{forward:}\quad y_A = x_A,\qquad y_B = x_B \odot e^{s_\theta(x_A)} + t_\theta(x_A),
\qquad \log|\det J_\psi(x)| = \textstyle\sum_j s_\theta(x_A)_j ;
$$

$$
\text{inverse:}\quad x_A = y_A,\qquad x_B = \big(y_B - t_\theta(y_A)\big) \odot e^{-s_\theta(y_A)},
\qquad \log|\det J_{\psi^{-1}}(y)| = -\textstyle\sum_j s_\theta(y_A)_j .
$$

Two names to fix once. `z` in the code is the **source-side** input of the layer, and
`x` the data-side output — in the chain of U1.T1 §3 these are $z_{k-1}$ and $z_k$; the
first layer's `z` is $z_0 = x_0$, the noise sample. `mask` is a boolean vector: `True`
marks $x_A$, `False` marks $x_B$.

**The interface is frozen** (artifact `coupling-layer-scratch`):

```
forward(z) -> (x, logdet_fwd)      # psi   : source -> data
inverse(x) -> (z, logdet_inv)      # psi^-1: data -> source
logdet_inv == -logdet_fwd, per example, at matching points
```

**The numerical policy** (U1.T1 §5, "a numerical aside for the lab"): the raw scale
output is **bounded** before exponentiation, $s = s_{\max}\tanh(\text{raw}/s_{\max})$
with $s_{\max} = 2$, so that $e^{s}$ can neither overflow nor divide the inverse by a
number near zero while the conditioner is still a random network. The notes measure
what happens when you forget this.

One more given decision, because it decides whether the checks below can fail at all.
The conditioner's last layer starts **small but nonzero**. A head initialized to
exactly zero gives $s \equiv 0$ at initialization, so the analytic log-determinant is
$0$, the autodiff log-determinant is $0$, and they agree perfectly on nothing. Two
bootcamp labs were caught by exactly that; this one is not.

### TODO 2.1 — `AffineCoupling`

The constructor and the conditioner are given. Write `_heads`, `forward` and
`inverse`.

In [ ]:
HEAD_INIT_SCALE = 0.1


def make_conditioner(d_in, d_out, hidden=(64, 64)):
    """d_in -> hidden ... -> d_out, SiLU inside, small NONZERO head (see above)."""
    dims = [d_in, *hidden]
    layers = []
    for a, b in zip(dims[:-1], dims[1:]):
        layers += [nn.Linear(a, b), nn.SiLU()]
    head = nn.Linear(dims[-1], d_out)
    with torch.no_grad():
        head.weight.mul_(HEAD_INIT_SCALE)
        head.bias.zero_()
    layers.append(head)
    return nn.Sequential(*layers)


class AffineCoupling(nn.Module):
    """One RealNVP affine coupling layer, generative direction psi: z -> x."""

    def __init__(self, d, mask, hidden=(64, 64), s_cap=2.0):
        super().__init__()
        mask = torch.as_tensor(mask, dtype=torch.bool)
        assert mask.shape == (d,) and mask.any() and (~mask).any()
        self.register_buffer("mask", mask)
        self.d_a, self.d_b = int(mask.sum()), d - int(mask.sum())
        self.s_cap = float(s_cap)
        self.cond = make_conditioner(self.d_a, 2 * self.d_b, hidden)  # heads: raw s, t

    def _heads(self, a):
        """The conditioner on the untouched half -> (s, t), s bounded by tanh."""
        # TODO 2.1a: split cond(a) into raw_s and t (torch.chunk), bound raw_s.
        raise NotImplementedError

    def forward(self, z):
        """(B, d) -> (x, logdet_fwd), logdet_fwd of shape (B,)."""
        # TODO 2.1b: a = z[:, mask], b = z[:, ~mask]; scale-and-shift b; sum s.
        raise NotImplementedError

    def inverse(self, x):
        """(B, d) -> (z, logdet_inv), logdet_inv = -sum s, shape (B,)."""
        # TODO 2.1c: same heads at x_A; undo shift, then scale; MINUS the sum.
        raise NotImplementedError

**Checkpoint 2.1** — shapes, and the one structural fact: the untouched half is
untouched.

In [ ]:
set_seed(0)
layer = AffineCoupling(2, [True, False])
z = torch.randn(8, 2)
x, ld = layer(z)
assert x.shape == (8, 2) and ld.shape == (8,)
assert torch.equal(x[:, 0], z[:, 0]), "x_A must pass through unchanged"
assert float(ld.abs().max()) > 1e-4, "log-det is ~0: the checks below would be vacuous"
print(f"Checkpoint 2.1 passed: x_A untouched; max |logdet| at init = {float(ld.abs().max()):.4f}")

### TODO 2.2 — The independent Jacobian witness

You wrote $\log|\det J_\psi| = \sum_j s_j$. Now let something that did **not** read
your derivation compute the same quantity. Automatic differentiation can assemble the
full $d \times d$ Jacobian of the map $z \mapsto \psi(z)$ **for one example**; its
`slogdet` is the number your sum must match. Do it in **float64**, on a small batch,
and batch the single-example Jacobian with `vmap`:

```python
J = torch.func.vmap(torch.func.jacrev(single_example_map))(z)   # (B, d, d)
sign, logabsdet = torch.linalg.slogdet(J)
```

**Do not** call `torch.autograd.functional.jacobian` on the whole batch and take one
determinant. That returns the Jacobian of the **batched map** $\mathbb{R}^{B d} \to
\mathbb{R}^{B d}$, a $(Bd)\times(Bd)$ object — not the $d\times d$ per-example matrix
you want. The cell after the checkpoint shows what it actually is.

Accept under the mixed tolerance of the ritual: `atol = 1e-10`, `rtol = 1e-8` on the
largest log-determinant in the batch.

In [ ]:
def per_example_logdet(forward_fn, z):
    """(B, d) -> (logabsdet (B,), sign (B,)) from PER-EXAMPLE autodiff Jacobians.

    `forward_fn` maps ONE example (d,) -> (d,). Call with a float64 `z` and a
    float64 (.double()) model.
    """
    # TODO 2.2: jacrev on the single-example map, vmap over the batch, slogdet.
    raise NotImplementedError


def witness(layer_or_flow, z, atol=1e-10, rtol=1e-8):
    """Analytic log-det vs autodiff log-det on the same float64 points."""
    with torch.no_grad():
        _, ld_analytic = layer_or_flow(z)

    def single(zi):
        return layer_or_flow(zi[None])[0][0]

    ld_ad, sign = per_example_logdet(single, z)
    gap = float((ld_ad.detach() - ld_analytic).abs().max())
    tol = atol + rtol * float(ld_analytic.abs().max())
    return gap, tol, bool((sign > 0).all())

**Checkpoint 2.2** — the witness on one layer, in float64. The reference solution
measures a gap near $5\times10^{-16}$ against a tolerance near $3\times10^{-10}$.

In [ ]:
set_seed(0)
layer64 = AffineCoupling(2, [True, False]).double()
z64 = torch.randn(64, 2, dtype=torch.float64)
gap, tol, positive = witness(layer64, z64)
print(f"analytic vs autodiff log-det: gap {gap:.2e}  (tol {tol:.2e})  det > 0: {positive}")
assert positive and gap <= tol
print("Checkpoint 2.2 passed.")

**The pitfall, made visible.** Here is the object the batched call returns. Its
off-diagonal blocks are exactly zero (example $i$ does not influence example $j$), so
its `slogdet` is the **sum** of the per-example log-determinants — one number for the
whole batch, which cannot tell you *which* example is wrong, and whose plain `det`
overflows float64 for a large batch of a trained flow.

In [ ]:
def batched(zz):
    return layer64(zz)[0]

J_batched = torch.autograd.functional.jacobian(batched, z64)      # (B, d, B, d)
B, d = z64.shape
J2 = J_batched.reshape(B * d, B * d)
off = J_batched.clone()
for i in range(B):
    off[i, :, i, :] = 0.0
with torch.no_grad():
    _, ld_each = layer64(z64)
print("shape of the batched Jacobian:", tuple(J_batched.shape))
print(f"largest off-diagonal block entry: {float(off.abs().max()):.1e}")
print(f"slogdet of the batched map: {float(torch.linalg.slogdet(J2)[1]):.10f}")
print(f"sum of per-example log-dets:  {float(ld_each.sum()):.10f}")

### TODO 2.3 — Both round trips, and the cancellation

Three checks, all per example, all in float64:

- `inverse(forward(z)) ≈ z`;
- `forward(inverse(x)) ≈ x`;
- `logdet_fwd + logdet_inv ≈ 0` at matching points.

In [ ]:
def round_trips(layer_or_flow, z):
    """Return (max |inverse(forward(z)) - z|, max |forward(inverse(x)) - x|,
    max |logdet_fwd + logdet_inv|) with x = forward(z)."""
    with torch.no_grad():
        # TODO 2.3: run forward, then inverse, then forward again; read both log-dets.
        raise NotImplementedError

In [ ]:
rt_z, rt_x, cancel = round_trips(layer64, z64)
print(f"z -> x -> z: {rt_z:.1e}   x -> z -> x: {rt_x:.1e}   logdet_fwd + logdet_inv: {cancel:.1e}")
assert rt_z < 1e-12 and rt_x < 1e-12 and cancel < 1e-12
print("Checkpoint 2.3 passed.")

## Block 3 — Stack, and train by exact maximum likelihood

### TODO 3.1 — `Flow`

A flow is a composition $\psi = \psi_K \circ \cdots \circ \psi_1$. Its inverse undoes
the layers in **reverse order**, and its log-determinant is the **sum** of the layer
log-determinants, each at the point that layer sees (U1.T1 §3). The contract, and the
equality that fixes the sign, written next to the code so that it is never implicit:

```
sample(n):    z ~ p_0, then forward           # x = psi(z)
log_prob(x):  z, logdet_inv = inverse(x)      # accumulate the inverse log-dets
              return log_p0(z) + logdet_inv
```

$$
\log p_\theta(x) \;=\; \log p_0(z) \;+\; \log\big|\det J_{\psi^{-1}}(x)\big|,
\qquad z = \psi^{-1}(x),
$$

which is U1.T1's $\log p_0(z_0) - \sum_k \log|\det J_{\psi_k}(z_{k-1})|$ with the
minus sign absorbed into `logdet_inv`. Masks **alternate**: layer $k$ reads the half
that layer $k-1$ moved, so every coordinate moves.

In [ ]:
LOG_2PI = math.log(2 * math.pi)


def log_p0(z):
    """log N(z; 0, I_d), per example."""
    return -0.5 * (z * z).sum(dim=-1) - 0.5 * z.shape[-1] * LOG_2PI


class Flow(nn.Module):
    def __init__(self, layers, d):
        super().__init__()
        self.layers = nn.ModuleList(layers)
        self.d = d

    @classmethod
    def realnvp(cls, d=2, n_layers=6, hidden=(64, 64), s_cap=2.0):
        base = torch.arange(d) % 2 == 0
        layers = [AffineCoupling(d, base if k % 2 == 0 else ~base, hidden, s_cap)
                  for k in range(n_layers)]
        return cls(layers, d)

    def forward(self, z):
        """psi: layers in order; log-dets ADD."""
        # TODO 3.1a
        raise NotImplementedError

    def inverse(self, x):
        """psi^-1: layers in REVERSE order, each one's .inverse; log-dets add."""
        # TODO 3.1b
        raise NotImplementedError

    def log_prob(self, x):
        """log p_0(psi^-1(x)) + log|det J_{psi^-1}(x)|, per example."""
        # TODO 3.1c
        raise NotImplementedError

    @torch.no_grad()
    def sample(self, n, generator=None):
        z = torch.randn(n, self.d, generator=generator)
        return self.forward(z)[0]

**Checkpoint 3.1** — four facts about the stack, and one about permutations.

1. `log_prob(x)` equals $\log p_0(z) - \texttt{logdet\_fwd}(z)$ at $z = \psi^{-1}(x)$
   (the sign, checked);
2. the witness passes on the whole six-layer stack;
3. both round trips pass;
4. **a permutation is free**: inserting a coordinate swap into the stack adds exactly
   $0$ to the log-determinant and undoes itself. U1.T1's Exercise 1 asked you to check
   this in your head; here it is in code. (`Permute` is given.)

In [ ]:
class Permute(nn.Module):
    """A fixed permutation of the coordinates. |det| = 1, so both log-dets are 0."""

    def __init__(self, perm):
        super().__init__()
        perm = torch.as_tensor(perm, dtype=torch.long)
        self.register_buffer("perm", perm)
        self.register_buffer("inv_perm", torch.argsort(perm))

    def forward(self, z):
        return z[:, self.perm], z.new_zeros(z.shape[0])

    def inverse(self, x):
        return x[:, self.inv_perm], x.new_zeros(x.shape[0])


set_seed(0)
flow64 = Flow.realnvp().double()
x64 = two_moons(64, 1).double()
with torch.no_grad():
    z64, ld_inv = flow64.inverse(x64)
    _, ld_fwd = flow64(z64)
    lp = flow64.log_prob(x64)
assert torch.allclose(lp, log_p0(z64) - ld_fwd, atol=1e-12), "log_prob sign"
gap, tol, positive = witness(flow64, z64)
assert positive and gap <= tol, (gap, tol)
rt_z, rt_x, cancel = round_trips(flow64, z64)
assert max(rt_z, rt_x, cancel) < 1e-12

mixed = Flow([flow64.layers[0], Permute([1, 0]), flow64.layers[1]], 2).double()
with torch.no_grad():
    y, ld_mixed = mixed(z64)
    _, ld_two = Flow([flow64.layers[0], flow64.layers[1]], 2).double()(z64)
    y_back, _ = mixed.inverse(y)
# the swap moves which half layer 2 reads, so ld_mixed != ld_two in general;
# what must hold is that the swap ITSELF contributed 0 and undoes itself.
assert float(Permute([1, 0]).double()(z64)[1].abs().max()) == 0.0
assert float((y_back - z64).abs().max()) < 1e-12
print(f"Checkpoint 3.1 passed: sign OK; stack witness gap {gap:.1e} (tol {tol:.1e}); "
      f"round trips {max(rt_z, rt_x):.1e}; permutation log-det 0.")

### TODO 3.2 — Train on two-moons with the given stack

The loss is U1.T1's, with the sampling subscript written out:

$$
\mathcal{L}(\theta) \;=\; -\,\mathbb{E}_{x_1 \sim q}\big[\log p_\theta(x_1)\big]
\;\approx\; -\frac{1}{B}\sum_{i=1}^{B} \log p_\theta\big(x_1^{(i)}\big).
$$

Everything about the loop is the given `Trainer`. You write the **closure** it needs:
a module whose `forward(x)` returns `log_prob(x)`, and a loss module that returns minus
its mean. The settings are the course defaults (AdamW at $3\times10^{-4}$, weight
decay $0.01$, linear warmup over 5% of the steps then cosine decay, stepped per step),
with $K = 6$ layers and conditioners of width 64.

In [ ]:
class LogProbModel(nn.Module):
    """The Trainer calls model(x); make that log p_theta(x)."""

    def __init__(self, flow):
        super().__init__()
        self.flow = flow

    def forward(self, x):
        # TODO 3.2a
        raise NotImplementedError


class NLL(nn.Module):
    """loss_fn(model(x), y) -> minus the mean log-density; y is a placeholder."""

    def forward(self, logp, y):
        # TODO 3.2b
        raise NotImplementedError

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fbgm-2026")


def warmup_cosine(step, total, warmup):
    """Multiplier on the base LR -- the U0.T2 default, stepped PER STEP."""
    if step < warmup:
        return step / max(1, warmup)
    p = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1.0 + math.cos(math.pi * p))


CFG = dict(n_layers=6, hidden=(64, 64), s_cap=2.0, n_train=4096, n_val=1024,
           seed_train=0, seed_val=1, batch_size=256, epochs=EPOCHS,
           lr=3e-4, weight_decay=0.01, warmup_frac=0.05, seed=0)


def fit_flow(model, gen, cfg, run_name):
    """The lab's training call: given Trainer, course defaults, MLflow run."""
    set_seed(cfg["seed"])
    xtr, xva = gen(cfg["n_train"], cfg["seed_train"]), gen(cfg["n_val"], cfg["seed_val"])
    tr = DataLoader(TensorDataset(xtr, torch.zeros(len(xtr))), batch_size=cfg["batch_size"],
                    shuffle=True, drop_last=True,
                    generator=torch.Generator().manual_seed(cfg["seed"]))
    va = DataLoader(TensorDataset(xva, torch.zeros(len(xva))), batch_size=1024)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    total = cfg["epochs"] * len(tr)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: warmup_cosine(s, total, int(cfg["warmup_frac"] * total)))
    trainer = Trainer(model, opt, NLL(), DEVICE, ckpt_dir=f"runs/{run_name}",
                      log_every=50, sched=sched, config=cfg)
    t0 = time.perf_counter()
    hist = trainer.fit(tr, va, cfg["epochs"])
    hist["seconds"] = time.perf_counter() - t0
    # Provenance and curves to MLflow, PER EPOCH, after the loop (see block 0
    # for why not per step here): params, tags, and the two NLL curves.
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({k: str(v) for k, v in cfg.items()})
        mlflow.set_tags({"session": "U1.L1", "data": gen.__name__, "seed": cfg["seed"],
                         "seconds": f"{hist['seconds']:.0f}"})
        for ep, (tl, vl) in enumerate(zip(hist["train_loss"], hist["val_loss"]), start=1):
            mlflow.log_metrics({"train_nll": tl, "val_nll": vl}, step=ep)
    return hist


set_seed(0)
flow_moons = Flow.realnvp(2, CFG["n_layers"], CFG["hidden"], CFG["s_cap"])
hist_moons = fit_flow(LogProbModel(flow_moons), two_moons, CFG, "u1l1-two-moons")
print(f"{hist_moons['seconds']:.0f} s on CPU")

**Checkpoint 3.2** — the loss decreased, and the model is still a bijection with the
right log-determinant. The criterion is **relative**: the validation NLL at the end
must beat the first epoch's by more than $1$ nat (the reference run improves by
$2.58$ nats, from $4.67$ to $2.10$; a half-length run still clears $1$ by a wide
margin). No absolute threshold, because an absolute number is a promise about your
hardware and configuration that nobody validated.

In [ ]:
v = hist_moons["val_loss"]
print(f"val NLL: first epoch {v[0]:.3f}  ->  last {v[-1]:.3f}  (best {min(v):.3f})")
assert v[-1] < v[0] - 1.0, "the validation NLL did not improve by 1 nat"

flow_moons.eval()
check = Flow.realnvp(2, CFG["n_layers"], CFG["hidden"], CFG["s_cap"]).double()
check.load_state_dict(flow_moons.state_dict())
with torch.no_grad():
    z_chk, _ = check.inverse(two_moons(64, 1).double())
gap, tol, positive = witness(check, z_chk)
rt_z, rt_x, cancel = round_trips(check, z_chk)
assert positive and gap <= tol and max(rt_z, rt_x, cancel) < 1e-10
print(f"Checkpoint 3.2 passed: witness gap {gap:.1e} (tol {tol:.1e}) after training; "
      f"round trips {max(rt_z, rt_x):.1e}.")

### TODO 3.3 — Same code, same budget, the gapped checkerboard

Nothing changes but the generator. Log the run under its own name and keep the model:
block 4 draws it.

In [ ]:
set_seed(0)
flow_checker = Flow.realnvp(2, CFG["n_layers"], CFG["hidden"], CFG["s_cap"])
# TODO 3.3: train flow_checker on `checkerboard` with fit_flow, run name "u1l1-checkerboard".
hist_checker = None
raise NotImplementedError

In [ ]:
v = hist_checker["val_loss"]
print(f"val NLL: first epoch {v[0]:.3f}  ->  last {v[-1]:.3f}  (best {min(v):.3f})")
print(f"the NLL of q itself (uniform on eight squares of side 1.5): log 18 = {math.log(18):.3f}")
assert v[-1] < v[0] - 1.0
print("Checkpoint 3.3 passed.")

## Block 4 — Look at what you built

Two pictures per model, both from `toy2d`'s helpers. The **density heatmap** evaluates
your `log_prob` on a grid — exact density evaluation, the thing this unit exists for.
The **warped grid** pushes a regular grid of the *source* space through $\psi^\theta$:
it is the visual twin of U1.T1's local-volume picture, and a cell that comes out large
is a place where the density was divided by a large $|\det J|$.

### TODO 4.1 — Two-moons: density beside the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
xd = two_moons(2000, 7).numpy()
axes[0].scatter(xd[:, 0], xd[:, 1], s=4, color="#6b7280", alpha=0.5, label="data")
xs_m = flow_moons.sample(2000, generator=torch.Generator().manual_seed(11)).numpy()
axes[0].scatter(xs_m[:, 0], xs_m[:, 1], s=4, color="#d97a1f", alpha=0.6, label="model")
axes[0].legend(markerscale=3)
# TODO 4.1: density_heatmap(<your log_prob>, ax=axes[1])
raise NotImplementedError
for ax, title in zip(axes, ("samples", "p_theta on the grid")):
    ax.set_xlim(*EXTENT); ax.set_ylim(*EXTENT); ax.set_aspect("equal"); ax.set_title(title)
plt.tight_layout(); plt.show()

### TODO 4.2 — Two-moons: the warped grid

`warped_grid` takes the **generative** map, $(B, 2) \to (B, 2)$. Your `forward` returns
a pair; pass a lambda that keeps the first element.

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 4.4))
# TODO 4.2: warped_grid(<generative map>, ax=ax)
raise NotImplementedError
ax.set_aspect("equal"); ax.set_title("the source grid under psi_theta")
plt.tight_layout(); plt.show()

### TODO 4.3 — The gapped checkerboard

Repeat both pictures for `flow_checker`, with the eight support squares outlined. Then
answer, in words, in the cell after the figure — and answer only what you can see:

> **What density does the model place in the hard-zero gaps, and what happens to the
> grid near those regions?**

`in_checkerboard_support` lets you put a number on the first half: the fraction of the
model's mass that lies outside the eight squares, and the smallest density along the
corner-to-corner segment between two diagonal neighbours.

In [ ]:
from matplotlib.patches import Rectangle
from common.toy2d import CHECKER_CELL, CHECKER_CENTRES, CHECKER_GAP

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
# TODO 4.3a: density_heatmap on axes[0], warped_grid on axes[1], for flow_checker
raise NotImplementedError
side = CHECKER_CELL - CHECKER_GAP
for ax in axes:
    for cx, cy in CHECKER_CENTRES:
        ax.add_patch(Rectangle((cx - side / 2, cy - side / 2), side, side,
                               fill=False, ls="--", lw=1.0, ec="#a3540c"))
    ax.set_xlim(*EXTENT); ax.set_ylim(*EXTENT); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

# TODO 4.3b: the numbers. Evaluate p_theta on a 400 x 400 grid; report the mass
# fraction outside the squares and the minimum density on the segment from
# (-2.25, -2.25) to (-1.75, -1.75) (two closest corners of diagonal neighbours).

**Your observation (TODO 4.3c, in words):**

## Block 5 — The library cross-check, and reading its source

### TODO 5.1 — Zuko's RealNVP, same data, same budget, same `Trainer`

Zuko is the course's reference implementation for discrete flows. Its flows are
`nn.Module`s whose call returns a **distribution**: `flow()` is $p_\theta$, with
`.log_prob` and `.sample`. So the closure of TODO 3.2 needs one change — `model(x)`
becomes `flow().log_prob(x)` — and **nothing else changes**: the data, the seeds, the
optimizer, the schedule, the `Trainer`. That is the point of the cross-check.

**The direction, translated.** Zuko parameterizes the *density* transform. In course
notation:

```
course:            psi          : z -> x        (sample: forward)
Zuko transform:    f = psi^-1   : x -> z        (flow.transform() maps DATA to SOURCE)
Zuko transform.inv = psi        : z -> x
```

`f.log_abs_det_jacobian(x, z)` is $\log|\det J_{\psi^{-1}}(x)|$ — your `logdet_inv`.
The checkpoint asserts this rather than trusting the sentence.

In [ ]:
import zuko

print("zuko", zuko.__version__)
zflow = zuko.flows.RealNVP(features=2, transforms=CFG["n_layers"],
                           hidden_features=CFG["hidden"])


class ZukoLogProb(nn.Module):
    def __init__(self, flow):
        super().__init__()
        self.flow = flow

    def forward(self, x):
        # TODO 5.1: the distribution is self.flow(); return its log_prob at x
        raise NotImplementedError


hist_zuko = fit_flow(ZukoLogProb(zflow), two_moons, CFG, "u1l1-zuko-two-moons")
zflow.eval()

In [ ]:
xv = two_moons(CFG["n_val"], CFG["seed_val"])
with torch.no_grad():
    f = zflow.transform()                      # the DENSITY transform f = psi^-1
    z_lib = f(xv)                              # data -> source
    ladj = f.log_abs_det_jacobian(xv, z_lib)   # log|det J_{psi^-1}(x)|
    lp_manual = log_p0(z_lib) + ladj
    lp_lib = zflow().log_prob(xv)
    x_back = f.inv(z_lib)                      # psi: source -> data
    nll_zuko = float(-lp_lib.mean())
    nll_scratch = float(-flow_moons.log_prob(xv).mean())
print(f"direction check |manual - library| log_prob: {float((lp_manual - lp_lib).abs().max()):.1e}")
print(f"f.inv(f(x)) round trip: {float((x_back - xv).abs().max()):.1e}")
print(f"validation NLL — scratch {nll_scratch:.3f}   Zuko {nll_zuko:.3f}")
assert float((lp_manual - lp_lib).abs().max()) < 1e-4
assert abs(nll_scratch - nll_zuko) < 0.15, "the two fits should land within 0.15 nat"

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
density_heatmap(flow_moons.log_prob, ax=axes[0])
density_heatmap(lambda pts: zflow().log_prob(pts), ax=axes[1])
for ax, title in zip(axes, (f"scratch ({nll_scratch:.3f})", f"Zuko RealNVP ({nll_zuko:.3f})")):
    ax.set_aspect("equal"); ax.set_title(title)
plt.tight_layout(); plt.show()
print("Checkpoint 5.1 passed.")

### TODO 5.2 — Source reading: your layer in Zuko's code

The anchors below were generated from **zuko 1.5.0**, the version pinned in the
course environment, when this notebook was built. Open the files (`import zuko;
zuko.__file__` gives the directory) and map each part of your `AffineCoupling` onto
the library. Zuko splits the layer into two classes: a **lazy** module that owns the
mask and the conditioner and builds a transform on demand, and the **transform** that
does the arithmetic.

| Your code | Zuko object | Where |
|---|---|---|
| the whole layer, lazily | `GeneralCouplingTransform` | `zuko/flows/coupling.py:25-139` |
| the arithmetic of one layer | `CouplingTransform` | `zuko/transforms.py:1001-1064` |
| the per-coordinate affine map $y = e^{a}x + b$ | `MonotonicAffineTransform` | `zuko/transforms.py:411-445` |
| the mask alternation across layers | `NICE.__init__` | `zuko/flows/coupling.py:164-196` |

Answer the four questions in the cell below the table (write in the markdown cell).

**(a) The mask.** Where is it stored, and which value marks the *untouched* half? Zuko's
docstring says: "Ones correspond to the constant split $x_a$". Compare with your
`mask` (True = $x_A$). Then find how `NICE.__init__` alternates it:
`mask = torch.arange(features) % 2 == i % 2` — see `zuko/flows/coupling.py:164-196`.

**(b) The conditioner (hypernetwork).** Find `self.hyper` in `zuko/flows/coupling.py:79-113`
and the `meta` method in `zuko/flows/coupling.py:128-136`. What does `meta` return, and what
plays the role of your `_heads`?

**(c) The inverse.** Read `_inverse` of the coupling transform at `zuko/transforms.py:1047-1051`
and of the affine map at `zuko/transforms.py:441-442`. In which direction is `_call`,
and in which is `_inverse`, in *course* notation?

**(d) The log-absolute-determinant path.** Read `log_abs_det_jacobian` at
`zuko/transforms.py:1053-1057` and `zuko/transforms.py:444-445`, and `call_and_ladj` at
`zuko/transforms.py:1059-1064`. Where is the sum over the moved coordinates taken? And
read the constructor at `zuko/transforms.py:425-436`: how does Zuko bound the log-scale,
and what is the bound for the default `slope = 1e-3`?

**Your answers (TODO 5.2):**

(a)

(b)

(c)

(d)

### TODO 5.3 — reserved

*Reserved landing zone: a deeper Zuko API and source exercise moves here only if
U1.L2 is cut from the calendar. Nothing to do today.*

## Block 6 — The same map in JAX (optional)

The JAX primer (U0.L5, in the self-study packet) introduced `jax.jacfwd` and
`jax.vmap`. Here is the whole generative coupling map in about fifteen lines of JAX,
loading **your trained weights**, and the per-example log-determinant witness in two
of them: `jax.vmap(jax.jacfwd(psi))` is the $(B, d, d)$ stack of per-example
Jacobians, exactly the object TODO 2.2 built with `torch.func`. The numbers must
agree with torch's to floating-point precision, because it is the same function with
the same weights. This block is skippable; nothing later depends on it.

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    jax.config.update("jax_enable_x64", True)
    HAVE_JAX = True
except ImportError:
    HAVE_JAX = False
    print("JAX not installed: skipping block 6.")

if HAVE_JAX:
    params = []
    for layer in flow_moons.layers:
        ws = [(jnp.asarray(m.weight.detach().double().numpy()),
               jnp.asarray(m.bias.detach().double().numpy()))
              for m in layer.cond if isinstance(m, nn.Linear)]
        params.append((bool(layer.mask[0]), ws, layer.s_cap))

    def mlp(ws, a):
        h = a
        for i, (w, b) in enumerate(ws):
            h = h @ w.T + b
            if i < len(ws) - 1:
                h = jax.nn.silu(h)
        return h

    def psi(z):                            # ONE example, (2,) -> (2,)
        x = z
        for first_is_a, ws, s_cap in params:
            ia, ib = (0, 1) if first_is_a else (1, 0)
            raw = mlp(ws, x[ia:ia + 1])
            s = s_cap * jnp.tanh(raw[0] / s_cap)
            x = x.at[ib].set(x[ib] * jnp.exp(s) + raw[1])
        return x

    xj = two_moons(64, 1).double()
    with torch.no_grad():
        zj, _ = check.inverse(xj)
        _, ld_torch = check(zj)
    J = jax.vmap(jax.jacfwd(psi))(jnp.asarray(zj.numpy()))     # (B, 2, 2)
    _, ld_jax = jnp.linalg.slogdet(J)
    gap = float(jnp.abs(ld_jax - jnp.asarray(ld_torch.numpy())).max())
    print(f"per-example log-det, JAX vs torch, same weights: max gap {gap:.1e}")
    assert gap < 1e-9

## Close — three objects you now own

| Object | Where | Checked by |
|---|---|---|
| a **bijection**, `AffineCoupling` + `Flow` | TODO 2.1, 3.1 | both round trips, per example (TODO 2.3) |
| **exact density evaluation**, `Flow.log_prob` | TODO 3.1 | the sign check and the training curve (Checkpoints 3.1–3.3) |
| an **independent Jacobian witness** | TODO 2.2 | autodiff per-example Jacobians in float64, before and after training |

And one thing you do not own yet: an explanation of the checkerboard's gaps. The
density you evaluated in them is real, the grid stretched across them is real, and the
half-nat gap to $\log 18$ is real. **What is exactly impossible for this model, and
what is merely hard, is the opening of U1.T2.**

### For PS1.4 (at home)

PS1.4(b) asked for $\log p_\theta(x_1)$ of a three-layer RealNVP in $d = 4$ as a fully
explicit expression. Your module can check your expression: build the three layers with
the masks of PS1.4 (layers 1 and 3 read $(\xi_1, \xi_2)$, layer 2 reads
$(\xi_3, \xi_4)$), evaluate your hand-written expression with each conditioner read at
the correct intermediate point, and compare with `Flow.log_prob`. The reference agrees
to $2\times10^{-15}$.

In [ ]:
set_seed(3)
mA = torch.tensor([True, True, False, False])
layers4 = [AffineCoupling(4, mA), AffineCoupling(4, ~mA), AffineCoupling(4, mA)]
flow4 = Flow(layers4, 4).double()
x1 = torch.randn(16, 4, dtype=torch.float64)
with torch.no_grad():
    z2, _ = layers4[2].inverse(x1)          # undo psi_3 first
    z1, _ = layers4[1].inverse(z2)
    z0, _ = layers4[0].inverse(z1)
    s3, _ = layers4[2]._heads(z2[:, mA])    # psi_3 reads (xi_1, xi_2) of z_2
    s2, _ = layers4[1]._heads(z1[:, ~mA])   # psi_2 reads (xi_3, xi_4) of z_1
    s1, _ = layers4[0]._heads(z0[:, mA])    # psi_1 reads (xi_1, xi_2) of z_0
    by_hand = log_p0(z0) - s1.sum(1) - s2.sum(1) - s3.sum(1)
    by_module = flow4.log_prob(x1)
print(f"PS1.4(b) by hand vs Flow.log_prob: {float((by_hand - by_module).abs().max()):.1e}")